# Lab 2 — The Lie Detector
**Session 2 · Prompt engineering + your first eval harness · TCE**

You'll need: your **10-question file from Lab 1 Part E**.
First: **File → Save a copy in Drive**.

In [ ]:
# Cell 1 — setup (same as Lab 1)
%pip install -q -U google-genai
from getpass import getpass
from google import genai
from google.genai import types
import time

MOCK = False   # ← set True only if the instructor says the live API is unavailable

client = genai.Client(api_key="mock" if MOCK else getpass("Gemini API key: "))
MODEL = "gemini-flash-lite-latest"  # lite: minimal thinking by default (near-zero thinking tokens on simple prompts), so the free tier goes further (Aug 2026 → Gemini 3.5 Flash Lite). 429 = rate limit: the helper below waits and retries; 503 = high demand: wait and re-run.

_MOCK_ANSWERS = [   # (keyword, canned reply) — offline path only. One fact below is deliberately wrong: the eval should catch it.
    ("pass or fail",     "PASS"),
    ("reply only a or b", "A"),                                   # a judge with a position bias — Stretch 3 will catch it
    ("roja",             "The music for Roja (1992) was composed by A. R. Rahman — it was his debut film score."),
    ("tce",              "Thiagarajar College of Engineering (TCE), Madurai, was founded in 1957 by Karumuttu Thiagarajan Chettiar. It is an autonomous institution affiliated to Anna University, known for its engineering programmes and its 100-acre campus on the Madurai–Theni road."),
    ("2011",             "India won the 2011 Cricket World Cup, beating Sri Lanka in the final at the Wankhede Stadium, Mumbai, under captain Virat Kohli."),   # ← wrong on purpose (it was Dhoni)
]
_MOCK_DEFAULT = "I am not sure."

def ask(prompt, temperature=None):
    if MOCK:
        p = prompt.lower()
        return "[MOCK] " + next((a for k, a in _MOCK_ANSWERS if k in p), _MOCK_DEFAULT)
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            # blocked/empty responses come back as None — treat as an empty answer, it scores as a miss
            return client.models.generate_content(model=MODEL, contents=prompt, config=config).text or ""
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited, waiting..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓  If any cell's import fails: Runtime -> Restart session, then re-run from this cell."
      + ("  (MOCK mode — canned answers)" if MOCK else ""))

## Part A — The prompt makeover (5 iterations)

Below is a deliberately terrible prompt. Improve it **five times**, one upgrade per run:
**v1** task (length+subject) → **v2** role+audience → **v3** context (real facts) → **v4** format+negative instructions → **v5** constraint ("only stated facts").

Run, read, then edit `PROMPT` and run again. Document each step in the table below.

The ten-question evaluation later is a learning instrument, not a reliable estimate of production accuracy. Keep evaluation questions separate from prompt examples, and keep them fixed while comparing prompts.

In [ ]:
# Cell 2 — edit PROMPT, run, repeat (keep old versions in comments!)
PROMPT = "write about tce"   # v0 — terrible on purpose

print(ask(PROMPT))

### Document your makeover (edit this cell)

| v | What I changed | Why the output got better |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### ✓ Checkpoint 1 — five documented iterations.

---
## Part B — Your test set → your first eval

Fill `my_tests` from your Lab 1 Part E file. Keep `expected` SHORT — the key fact only (a name, a number), not a full sentence. The scorer checks whether your expected string appears inside the model's answer (after normalization).

In [ ]:
# Cell 3 — your 10 questions (3 examples shown — replace with YOURS)
my_tests = [
    {"q": "Who composed the music for the film Roja?",                     "expected": "rahman"},
    {"q": "In which year was TCE Madurai founded?",                         "expected": "1957"},
    {"q": "Who captained India in the 2011 Cricket World Cup final?",       "expected": "dhoni"},
    # ... add your 7+ more ...
]
print(len(my_tests), "questions loaded")

In [ ]:
# Cell 4 — the eval harness
import re

def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower())

def run_eval(template, tests, verbose=True):
    if not tests:
        raise ValueError("Add at least one labelled test before running the evaluation")
    hits = 0
    for t in tests:
        ans = ask(template.format(q=t["q"]), temperature=0.0)
        ok = norm(t["expected"]) in norm(ans)
        hits += ok
        if verbose:
            print(("✓" if ok else "✗"), t["q"])
            if not ok:
                print("   expected:", t["expected"], "| got:", ans[:120].replace("\n"," "))
    score = hits / len(tests)
    print(f"SCORE: {hits}/{len(tests)} = {score:.0%} (learning set size n={len(tests)})")
    print("Interpret this as evidence about these questions, not a production accuracy claim.")
    return score

baseline = run_eval("Answer this question: {q}", my_tests)

### Read every ✗ before moving on
For each failure, decide: **model wrong** (hallucination — the interesting case), **scorer too strict** (fix your `expected` string), or **question ambiguous** (fix the question). This diagnosis IS the skill.

**Scored 9 or 10 out of 10?** Your set is too easy to measure anything — swap in 5 obscure questions (deep cuts, not headlines) before Part C.

### ✓ Checkpoint 2 — eval ran on your 10 questions; failures diagnosed.

---
## Part C — Prompt A vs Prompt B, settled with numbers

In [ ]:
# Cell 5 — design a better template, then fight (on the first 5 questions while you iterate)
PROMPT_A = "Answer this question: {q}"

# Only {q} may appear in braces — a literal { } in your template will crash .format(). Ask for 'JSON with keys name, degree, year' in words instead.
PROMPT_B = (
    "You are a careful expert. Answer the question below.\n"
    "Rules: be direct, give the specific fact asked for, "
    "and if you are not sure, say 'I am not sure' instead of guessing.\n\n"
    "Question: {q}\nAnswer:"
)   # ← edit me — beat A by more!

DEV = my_tests[:5]     # iterate here: 5 calls per prompt per run, not 10 — you have another lab today
print("=== A ==="); score_a = run_eval(PROMPT_A, DEV, verbose=False)
print("=== B ==="); score_b = run_eval(PROMPT_B, DEV, verbose=False)
print(f"\nA: {score_a:.0%}  vs  B: {score_b:.0%} on n={len(DEV)}  →  {'B wins' if score_b>score_a else 'A wins or tie — iterate B!'}")

In [ ]:
# Cell 5b — CONFIRMATION RUN on all 10 (run ONCE, when PROMPT_B is final — 20 calls)
# A win on 5 questions is a hint, not a result. Confirm on the full set before you claim it.
print("=== A (all) ==="); score_a_all = run_eval(PROMPT_A, my_tests, verbose=False)
print("=== B (all) ==="); score_b_all = run_eval(PROMPT_B, my_tests, verbose=True)     # verbose: read every ✗
print(f"\nCONFIRMED  A: {score_a_all:.0%}  vs  B: {score_b_all:.0%} on n={len(my_tests)}")
print("Report the numbers WITH n. Did the 5-question winner survive the 10-question run?")

### ✓ Checkpoint 3 — show me A vs B numbers + the single most interesting failure you found.

---
## Stretch goals

In [ ]:
# Stretch 1 — variance: is your score stable?
scores = [run_eval(PROMPT_B, my_tests, verbose=False) for _ in range(3)]
print("three runs:", [f"{s:.0%}" for s in scores], "| average:", f"{sum(scores)/3:.0%}")
# We used temperature=0.0 in the harness — try changing it in run_eval and watch stability change.

In [ ]:
# Stretch 2 — LLM-as-judge (a model grades a model)
def judge_score(question, expected, answer):
    verdict = ask(
        f"Question: {question}\nExpected key fact: {expected}\nStudent answer: {answer}\n"
        "Does the student answer contain the expected fact (paraphrase ok)? Reply only PASS or FAIL.",
        temperature=0.0)
    return "PASS" in verdict.upper()

t = my_tests[0]
ans = ask(PROMPT_B.format(q=t["q"]))
print("judge says:", judge_score(t["q"], t["expected"], ans))
# Now: where might the judge itself be wrong? (verbosity bias, self-agreement...)

In [ ]:
# Stretch 3 — position bias: a PAIRWISE judge, run both ways
# A judge that compares two answers can prefer whichever comes first. Swap the order and see how often it changes its mind.
def judge_pair(question, a, b):
    """Returns 'A' or 'B' — which of the two answers the judge prefers."""
    v = ask(f"Question: {question}\n\nAnswer A: {a}\n\nAnswer B: {b}\n\n"
            "Which answer is better — more correct, more specific? Reply only A or B.", temperature=0.0)
    v = v.removeprefix("[MOCK] ").strip().upper()
    return "B" if v.startswith("B") else "A"

flips = 0
for t in my_tests[:3]:                                   # 3 questions × (2 answers + 2 judgements) = 12 calls
    a, b = ask(PROMPT_A.format(q=t["q"])), ask(PROMPT_B.format(q=t["q"]))
    first  = judge_pair(t["q"], a, b)                    # A shown first
    second = judge_pair(t["q"], b, a)                    # B shown first — so 'A' here means B won
    second = "B" if second == "A" else "A"               # map back to the real labels
    flips += first != second
    print(f"{'DISAGREES' if first != second else 'agrees  '}  A-first says {first}, B-first says {second}  | {t['q'][:50]}")
print(f"\nposition-bias disagreements: {flips}/3  — every disagreement is a verdict decided by ORDER, not content")

## Wrap
You now own the loop: **prompt → eval → read failures → fix → re-run.** Keep this notebook — the same harness grades your capstone in Session 6.

Before you close: **File → Save a copy in Drive** again (your test set only exists in this notebook), and paste `my_tests` into a text file too.

**Short break. Session 3: AI gets eyes and ears — have a photo or two on your phone.**